# Module 05 — Exercises

These exercises make you implement the pieces of Module 05 yourself: the stability limit of gradient
descent, mini-batch SGD, a convexity test, learning-rate schedules, Adam from its equations, Newton's
method, a Lagrange problem, gradient clipping and a learning-rate sweep, two conceptual questions and
a mini-project comparing optimisers on a classification task. Use only NumPy (and the imports in the
setup cell) unless an exercise says otherwise.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Gradient descent on a quadratic and the largest stable step  `★☆☆`

For $L(\theta) = \tfrac12\theta^\top A\theta$ with symmetric positive-definite $A$, gradient descent is
$\theta_{k+1} = \theta_k - \eta A\theta_k$. Implement

* `gd_quadratic(A, theta0, eta, steps)` — returns the NumPy array of iterates of shape
  `(steps + 1, d)`, starting with `theta0`;
* `max_stable_step(A)` — returns the largest step size for which GD converges from every start,
  as a `float` ($2/L$ with $L$ the largest eigenvalue; use `np.linalg.eigvalsh`);
* `optimal_step(A)` — returns the step $2/(L + \mu)$ that minimises the worst-case contraction factor.

Test matrix: `A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])`.

In [ ]:
def gd_quadratic(A, theta0, eta, steps):
    # YOUR CODE HERE
    return None

def max_stable_step(A):
    # YOUR CODE HERE
    return None

def optimal_step(A):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
try:
    path = gd_quadratic(A, np.array([1., 1., 1.]), 0.1, 50)
    shape_ok = path.shape == (51, 3)
    converged = np.linalg.norm(path[-1]) < 1e-2 * np.linalg.norm(path[0])
except Exception:
    shape_ok, converged = False, False
check('gd_quadratic returns 51 iterates of dimension 3', shape_ok)
check('with eta = 0.1 the iterates shrink by 100x in 50 steps', converged)
try:
    eta_max = max_stable_step(A)
    diverges = np.linalg.norm(gd_quadratic(A, np.array([1., 1., 1.]), 1.1 * eta_max, 60)[-1]) > 10
    stays = np.linalg.norm(gd_quadratic(A, np.array([1., 1., 1.]), 0.9 * eta_max, 60)[-1]) < 1
except Exception:
    eta_max, diverges, stays = None, False, False
check_hash('hidden: largest stable step 2/L for the test matrix', eta_max, '1632b097f34d9c16')
check('GD at 1.1 * max_stable_step diverges, at 0.9 * max_stable_step it does not', diverges and stays)
try:
    eta_opt = optimal_step(A)
except Exception:
    eta_opt = None
check_hash('hidden: optimal constant step 2/(L+mu)', eta_opt, '9b7483b9cce48380')

## Exercise 2 — Convexity test via Hessian eigenvalues  `★☆☆`

A twice-differentiable function is convex on a region iff its Hessian is positive semi-definite at
every point of that region. Implement `min_curvature(hess_fn, grid)` which takes a function
`hess_fn(x, y)` returning the $2\times2$ Hessian at a point and an iterable of `(x, y)` points, and
returns the **smallest Hessian eigenvalue over the whole grid** as a `float` (use `np.linalg.eigvalsh`).
A negative result proves the function is not convex on that region.

Then write the Hessians by hand for

* $f(x, y) = x^4 - 2x^2 + y^2 + xy$ as `hess_f(x, y)`,
* $g(x, y) = x^2 + xy + y^2$ as `hess_g(x, y)`,

and evaluate `min_curvature` for both on the grid `[(x, y) for x in np.linspace(-2, 2, 41) for y in np.linspace(-2, 2, 41)]`.

In [ ]:
def min_curvature(hess_fn, grid):
    # YOUR CODE HERE
    return None

def hess_f(x, y):
    # YOUR CODE HERE  (Hessian of x^4 - 2x^2 + y^2 + xy)
    return None

def hess_g(x, y):
    # YOUR CODE HERE  (Hessian of x^2 + xy + y^2)
    return None

**Check your answer:**

In [ ]:
grid = [(x, y) for x in np.linspace(-2, 2, 41) for y in np.linspace(-2, 2, 41)]
try:
    hf0 = np.asarray(hess_f(0.0, 0.0), dtype=float); hg0 = np.asarray(hess_g(0.5, -1.0), dtype=float)
except Exception:
    hf0, hg0 = None, None
check_close('hess_f at the origin is [[-4, 1], [1, 2]]', hf0, np.array([[-4., 1.], [1., 2.]]))
check_close('hess_g is the constant matrix [[2, 1], [1, 2]]', hg0, np.array([[2., 1.], [1., 2.]]))
try:
    mf = min_curvature(hess_f, grid); mg = min_curvature(hess_g, grid)
except Exception:
    mf, mg = None, None
check_hash('hidden: smallest curvature of f on the grid', mf, '197ed5f20d94ef69')
check_hash('hidden: smallest curvature of g on the grid', mg, '3cae67a480a37535')
check('f is NOT convex on the grid, g is', (mf is not None) and (mg is not None) and mf < 0 < mg)

## Exercise 3 — Learning-rate schedules  `★☆☆`

Implement two schedules as plain functions of the step counter $t$ (integers starting at 0):

* `cosine_lr(t, T, eta_max, eta_min=0.0)` $= \eta_{\min} + \tfrac12(\eta_{\max} - \eta_{\min})\big(1 + \cos(\pi t/T)\big)$ for $0 \le t \le T$;
* `warmup_cosine(t, T_warm, T, eta_max)` — linear warm-up $\eta_{\max}\, t / T_{\text{warm}}$ for
  $t < T_{\text{warm}}$, then cosine decay from $\eta_{\max}$ to $0$ over the remaining $T - T_{\text{warm}}$
  steps (i.e. `cosine_lr(t - T_warm, T - T_warm, eta_max, 0.0)`).

Both return a `float`.

In [ ]:
def cosine_lr(t, T, eta_max, eta_min=0.0):
    # YOUR CODE HERE
    return None

def warmup_cosine(t, T_warm, T, eta_max):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    c = [cosine_lr(t, 1000, 1.0) for t in (0, 250, 500, 750, 1000)]
except Exception:
    c = None
check_close('cosine: eta at t = 0, T/4, T/2, 3T/4, T', c, [1.0, 0.8535533906, 0.5, 0.1464466094, 0.0])
try:
    wc = [warmup_cosine(t, 100, 1000, 0.1) for t in (0, 50, 100, 1000)]
    total = float(sum(warmup_cosine(t, 100, 1000, 0.1) for t in range(1001)))
except Exception:
    wc, total = None, None
check_close('warm-up + cosine: eta at t = 0, 50, 100, 1000', wc, [0.0, 0.05, 0.1, 0.0])
check_hash('hidden: total learning rate summed over t = 0..1000', total, '30ca21619217583f')

## Exercise 4 — Mini-batch SGD for linear regression  `★★☆`

Implement `sgd_linreg(X, y, eta, batch_size, epochs, seed)` for the mean-squared-error loss
$L(w) = \tfrac1n\lVert Xw - y\rVert^2$ (no intercept, $w$ starts at zeros). Follow this protocol exactly
so that your result matches the reference run:

1. `r = np.random.default_rng(seed)`;
2. for each epoch: `perm = r.permutation(n)`, then for `s` in `range(n // batch_size)` take the
   rows `perm[s*batch_size:(s+1)*batch_size]` as the batch (drop the remainder);
3. update `w = w - eta * g` with the mini-batch gradient $g = \tfrac{2}{b}X_B^\top(X_B w - y_B)$.

Return the final `w` (shape `(d,)`). Data for the check:

```python
r = np.random.default_rng(7)
X = r.normal(size=(500, 4)) * np.array([1.0, 2.0, 0.5, 1.5])
w_true = np.array([2.0, -1.0, 0.5, 3.0])
y = X @ w_true + 0.3 * r.normal(size=500)
```

In [ ]:
def sgd_linreg(X, y, eta, batch_size, epochs, seed):
    r = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
r = np.random.default_rng(7)
X = r.normal(size=(500, 4)) * np.array([1.0, 2.0, 0.5, 1.5])
w_true = np.array([2.0, -1.0, 0.5, 3.0])
y = X @ w_true + 0.3 * r.normal(size=500)
w_ols = np.linalg.lstsq(X, y, rcond=None)[0]
try:
    w_hat = sgd_linreg(X, y, 0.05, 25, 5, 0)
    close_to_ols = np.abs(w_hat - w_ols).max() < 0.1
except Exception:
    w_hat, close_to_ols = None, False
check('after 5 epochs the SGD solution is within 0.1 of the least-squares solution', close_to_ols)
check_close('exact match with the reference run (same seed, same protocol)', w_hat, np.array([ 1.9972322919, -0.9724221034,  0.4132140561,  3.0040908717]), tol=1e-6)
try:
    w_full = sgd_linreg(X, y, 0.2, 500, 300, 0)           # batch = n is plain gradient descent (eta = 0.2 < 2/L = 0.255)
except Exception:
    w_full = None
check_close('with batch_size = n it is gradient descent and converges to OLS', w_full, w_ols, tol=1e-3)

## Exercise 5 — Adam from the update equations  `★★☆`

Implement one Adam step as a pure function

```python
adam_step(theta, g, m, v, t, eta=1e-3, beta1=0.9, beta2=0.999, eps=1e-8) -> (theta_new, m_new, v_new)
```

following the equations $m \leftarrow \beta_1 m + (1-\beta_1) g$, $v \leftarrow \beta_2 v + (1-\beta_2) g^2$
(element-wise), $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$,
$\theta \leftarrow \theta - \eta\,\hat m/(\sqrt{\hat v} + \epsilon)$. The step counter `t` starts at 1.

Then use it to minimise the quadratic $L(\theta) = \tfrac12\theta^\top A\theta - b^\top\theta$ with
`A = np.array([[3., 1.], [1., 1.]])`, `b = np.array([1., -2.])`, gradient $A\theta - b$, from
$\theta_0 = 0$, with `eta = 0.1` for exactly 200 steps; store the result in `theta_adam`.

In [ ]:
def adam_step(theta, g, m, v, t, eta=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
    # YOUR CODE HERE
    return None, None, None

A = np.array([[3., 1.], [1., 1.]]); b = np.array([1., -2.])
theta_adam = None
# YOUR CODE HERE: run 200 Adam steps with eta = 0.1 from theta = zeros(2)

**Check your answer:**

In [ ]:
try:
    th1, m1, v1 = adam_step(np.zeros(2), np.array([0.1, -100.0]), np.zeros(2), np.zeros(2), 1, eta=0.01)
except Exception:
    th1, m1, v1 = None, None, None
check_close('first step has size eta in every coordinate, whatever the gradient scale', th1, np.array([-0.01, 0.01]), tol=1e-6)
check_close('first-moment state after step 1 is (1 - beta1) * g', m1, np.array([0.01, -10.0]), tol=1e-9)
check_close('second-moment state after step 1 is (1 - beta2) * g^2', v1, np.array([1e-5, 10.0]), tol=1e-9)
try:
    got = np.asarray(theta_adam, dtype=float)
except Exception:
    got = None
check_close('200 Adam steps on the quadratic match the reference run', got, np.array([ 1.5000067453, -3.5000123330]), tol=1e-6)
check_close('... and land near the exact minimiser A^-1 b = (1.5, -3.5)', got, np.linalg.solve(np.array([[3., 1.], [1., 1.]]), np.array([1., -2.])), tol=1e-3)

## Exercise 6 — Newton's method for logistic regression  `★★☆`

For the $\ell_2$-regularised logistic loss (labels $y_i \in \{0, 1\}$, first column of $X$ is the intercept,
which is **not** penalised)

$$
L(w) = \frac1n\sum_{i=1}^n \Big[-y_i \log p_i - (1-y_i)\log(1-p_i)\Big] + \frac{\lambda}{2}\sum_{j \ge 1} w_j^2,
\qquad p_i = \sigma(x_i^\top w),
$$

the gradient is $\nabla L = \tfrac1n X^\top(p - y) + \lambda \tilde w$ and the Hessian is
$\nabla^2 L = \tfrac1n X^\top S X + \lambda \tilde I$, where $S = \mathrm{diag}\big(p_i(1-p_i)\big)$ and
$\tilde w$, $\tilde I$ have a zero in the intercept position. Implement

* `logreg_grad_hess(w, X, y, lam)` returning the tuple `(grad, hess)`;
* `newton_logreg(X, y, lam, iters)` which starts at `w = 0` and performs `iters` Newton steps
  `w = w - solve(H, g)`, returning `w`.

Use `np.linalg.solve`, not an explicit inverse. The check runs 10 iterations with $\lambda = 10^{-2}$ on the
standardised breast-cancer data (with an intercept column) and verifies that the gradient has
vanished to $10^{-8}$ — the quadratic-convergence signature.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def logreg_grad_hess(w, X, y, lam):
    # YOUR CODE HERE
    return None, None

def newton_logreg(X, y, lam, iters):
    n, d = X.shape
    w = np.zeros(d)
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
data = load_breast_cancer()
Xr, yb = data.data, data.target.astype(float)
Xs = np.c_[np.ones(len(Xr)), (Xr - Xr.mean(0)) / Xr.std(0)]
try:
    g0, H0 = logreg_grad_hess(np.zeros(Xs.shape[1]), Xs, yb, 1e-2)
    g0 = np.asarray(g0, dtype=float); H0 = np.asarray(H0, dtype=float)
    h_ok = H0.shape == (31, 31) and np.allclose(H0, H0.T) and np.allclose(H0[1:, 1:] - 1e-2 * np.eye(30), 0.25 * Xs[:, 1:].T @ Xs[:, 1:] / len(yb))
    h_intercept = abs(H0[0, 0] - 0.25) < 1e-9
except Exception:
    g0, h_ok, h_intercept = None, False, False
check_close('gradient at w = 0 is X^T (0.5 - y) / n', g0, Xs.T @ (0.5 - yb) / len(yb), tol=1e-9)
check('Hessian at w = 0 is X^T X / (4n) plus lambda on the penalised block', h_ok)
check('the intercept is not penalised (H[0, 0] = 0.25 at w = 0)', h_intercept)
try:
    w_hat = newton_logreg(Xs, yb, 1e-2, 10)
    g_fin, _ = logreg_grad_hess(w_hat, Xs, yb, 1e-2)
    gnorm = float(np.linalg.norm(g_fin))
    acc = float(np.mean((Xs @ w_hat > 0) == (yb == 1)))
    loss_fin = float(np.mean(yb * np.logaddexp(0, -Xs @ w_hat) + (1 - yb) * np.logaddexp(0, Xs @ w_hat)) + 0.5 * 1e-2 * (w_hat[1:] @ w_hat[1:]))
except Exception:
    gnorm, acc, loss_fin = None, None, None
check('after 10 Newton steps the gradient norm is below 1e-8', gnorm is not None and gnorm < 1e-8)
check_hash('hidden: training accuracy of the Newton solution', acc, 'a8615c3feb3f99e3')
check_hash('hidden: regularised loss at the optimum (3 decimals)', loss_fin, 'cc1a4126ba1a738f')

## Exercise 7 — Lagrange multipliers and KKT by hand  `★★☆`

Solve the following two problems **analytically** (pen and paper, then type in the numbers):

**(a)** Minimise $f(x, y, z) = x^2 + y^2 + z^2$ subject to $x - 2y + 3z = 14$. Report the minimiser
as `x_star_a` (NumPy array of length 3), the optimal value `f_star_a`, and the multiplier
`lambda_a` in the convention $\nabla f + \lambda\nabla h = 0$ with $h = x - 2y + 3z - 14$.

**(b)** Minimise $g(x, y) = (x - 3)^2 + (y - 1)^2$ subject to $x + y \le 2$. Report the minimiser
`x_star_b` (array of length 2) and the KKT multiplier `mu_b` $\ge 0$ of the constraint
$x + y - 2 \le 0$ (stationarity: $\nabla g + \mu\nabla c = 0$).

Hint for (a): stationarity gives $x \propto \nabla h$; for (b) first decide whether the constraint
is active by checking the unconstrained minimiser.

In [ ]:
x_star_a = None      # np.array([...])
f_star_a = None
lambda_a = None

x_star_b = None      # np.array([...])
mu_b = None

**Check your answer:**

In [ ]:
try:
    xa = np.asarray(x_star_a, dtype=float)
    feasible_a = abs(xa @ np.array([1., -2., 3.]) - 14.0) < 1e-6
    parallel_a = np.allclose(np.cross(2 * xa, np.array([1., -2., 3.])), 0.0, atol=1e-6)
except Exception:
    xa, feasible_a, parallel_a = None, False, False
check('(a) your point satisfies the constraint', feasible_a)
check('(a) grad f is parallel to grad h at your point', parallel_a)
check_hash('(a) hidden: minimiser', (xa if xa is not None else None), 'db5afbf1a5b26ebb')
check_hash('(a) hidden: optimal value', (float(f_star_a) if f_star_a is not None else None), '5975a5ae2613ac58')
check_hash('(a) hidden: multiplier lambda', (float(lambda_a) if lambda_a is not None else None), 'e8abed6037ff1efd')
try:
    xb = np.asarray(x_star_b, dtype=float)
    active_b = abs(xb.sum() - 2.0) < 1e-6
except Exception:
    xb, active_b = None, False
check('(b) the constraint is active at your point', active_b)
check_hash('(b) hidden: minimiser', (xb if xb is not None else None), 'e4e39badc1c13a10')
check_hash('(b) hidden: multiplier mu', (float(mu_b) if mu_b is not None else None), 'ebda82fe4e8dbddb')
check('(b) dual feasibility: mu >= 0', mu_b is not None and float(mu_b) >= 0)

## Exercise 8 — Gradient clipping and a learning-rate sweep  `★★☆`

Two diagnostics from Section 8 of the guide.

1. `clip_by_global_norm(g, c)` — return $g \cdot \min(1, c/\lVert g\rVert_2)$ (and `g` unchanged if its norm is 0).
2. `lr_sweep(A, x0, etas, steps=30)` — for each $\eta$ in `etas`, run `steps` iterations of GD on
   $\tfrac12 x^\top A x$ from `x0` and return the array of final loss values $\tfrac12 x^\top A x$, one per $\eta$.

Then, with `A = np.diag([1., 10.])`, `x0 = np.array([10., 1.])` and `etas = np.logspace(-2, 0, 41)`,
set `best_eta` to the $\eta$ with the smallest final loss and `first_bad_eta` to the smallest $\eta$ at which the
final loss **exceeds the initial loss** $\tfrac12 x_0^\top A x_0$ (divergence). Compare both with $2/L$ and $2/(L+\mu)$.

In [ ]:
def clip_by_global_norm(g, c):
    # YOUR CODE HERE
    return None

def lr_sweep(A, x0, etas, steps=30):
    # YOUR CODE HERE
    return None

A = np.diag([1., 10.]); x0 = np.array([10., 1.]); etas = np.logspace(-2, 0, 41)
best_eta = None
first_bad_eta = None
# YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    c1 = clip_by_global_norm(np.array([3.0, 4.0]), 1.0)
    c2 = clip_by_global_norm(np.array([0.3, 0.4]), 1.0)
    c3 = clip_by_global_norm(np.zeros(2), 1.0)
except Exception:
    c1, c2, c3 = None, None, None
check_close('clipping (3, 4) at c = 1 gives (0.6, 0.8)', c1, np.array([0.6, 0.8]))
check_close('a gradient already inside the ball is unchanged', c2, np.array([0.3, 0.4]))
check_close('the zero gradient stays zero', c3, np.zeros(2))
try:
    sw = np.asarray(lr_sweep(np.diag([1., 10.]), np.array([10., 1.]), np.array([0.05, 0.1, 0.3]), steps=5), dtype=float)
except Exception:
    sw = None
check_close('lr_sweep: 5 steps at eta = 0.05, 0.1, 0.3', sw, np.array([  29.9417297744,   17.4339220050, 5121.4123762450]), tol=1e-6)
check_hash('hidden: best eta on the grid', (float(best_eta) if best_eta is not None else None), 'a77d78c4ef018ce3')
check_hash('hidden: first diverging eta on the grid', (float(first_bad_eta) if first_bad_eta is not None else None), '4583af2b9144175a')
check('best eta is below 2/L = 0.2 and first diverging eta is above it', best_eta is not None and first_bad_eta is not None and best_eta < 0.2 < first_bad_eta)

## Exercise 9 — Heavy-ball momentum beats gradient descent by a factor of about sqrt(kappa)  `★★★`

On the quadratic $\tfrac12 x^\top A x$ with $A = \mathrm{diag}(1, \kappa)$, $\kappa = 100$, compare

* `steps_to_tol_gd(A, x0, eta, tol=1e-6, max_steps=100000)` — plain GD; return the number of steps
  $k$ (counting from 1) after which $\lVert x_k\rVert < $ `tol` for the first time (or `None`);
* `heavy_ball(A, x0, eta, beta, tol=1e-6, max_steps=100000)` — Polyak momentum
  $v_k = \beta v_{k-1} + A x_{k-1}$, $x_k = x_{k-1} - \eta v_k$ with $v_0 = 0$; same return convention.

Run both from `x0 = np.array([1., 1.])`: GD with its optimal step $\eta = 2/(L+\mu)$, and heavy ball with
the tuned constants $\beta = \big(\tfrac{\sqrt\kappa - 1}{\sqrt\kappa + 1}\big)^2$,
$\eta = \big(\tfrac{2}{\sqrt L + \sqrt\mu}\big)^2$. Store the two step counts in `k_gd` and `k_hb`. The
theory predicts rates $(\kappa-1)/(\kappa+1) = 0.980$ versus $(\sqrt\kappa-1)/(\sqrt\kappa+1) = 0.818$.

In [ ]:
def steps_to_tol_gd(A, x0, eta, tol=1e-6, max_steps=100000):
    # YOUR CODE HERE
    return None

def heavy_ball(A, x0, eta, beta, tol=1e-6, max_steps=100000):
    # YOUR CODE HERE
    return None

kappa = 100.0; A = np.diag([1.0, kappa]); x0 = np.array([1.0, 1.0])
k_gd = None
k_hb = None
# YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    k1 = steps_to_tol_gd(np.diag([1., 1.]), np.array([1., 1.]), 0.5, tol=1e-6)      # identity: factor 0.5 per step
    k2 = heavy_ball(np.diag([1., 1.]), np.array([1., 1.]), 0.5, 0.0, tol=1e-6)      # beta = 0 is plain GD
except Exception:
    k1, k2 = None, None
check('GD on the identity with eta = 0.5 needs 21 steps to shrink below 1e-6', k1 == 21)
check('heavy ball with beta = 0 reproduces GD', k2 == 21)
check_hash('hidden: GD steps for kappa = 100', (int(k_gd) if k_gd is not None else None), '92c5fd0421c1d619')
check_hash('hidden: heavy-ball steps for kappa = 100', (int(k_hb) if k_hb is not None else None), 'ad48ff99415b2f00')
check('heavy ball is at least 5x faster', k_gd is not None and k_hb is not None and k_gd / k_hb > 5)

## Exercise 10 — Why momentum helps in narrow valleys  `★★☆`

A colleague says: *"Momentum just makes the effective learning rate $\eta/(1-\beta)$ larger. So I could
get the same effect by using plain gradient descent with a 10x larger step."* Explain, using the
ill-conditioned bowl with eigenvalues $\mu = 1$ and $L = 100$, (i) what goes wrong with the 10x
larger plain step, (ii) why momentum does not suffer from the same problem, and (iii) roughly how
many steps each method needs to reduce the error by $10^{6}$.

<details><summary><b>Show answer</b></summary>

(i) The largest stable step for GD is $2/L = 0.02$. A step of $10/L = 0.1$ multiplies the stiff
coordinate by $1 - 0.1\cdot100 = -9$ per iteration: it diverges immediately. The effective step of
momentum is only $\eta/(1-\beta)$ for a *constant* gradient; on an oscillating one it is much smaller.

(ii) In the stiff direction the gradient flips sign at every step, so the velocity
$v_t = \beta v_{t-1} + g_t$ keeps *cancelling* and stays small; in the flat direction the gradient
always points the same way, so $v_t$ builds up to $g/(1-\beta)$. Momentum therefore behaves like a
large step along the valley and a small step across it — something no single scalar $\eta$ can do.

(iii) GD with its best constant step contracts by $(\kappa-1)/(\kappa+1) = 0.980$ per step, so
$10^{6}$ needs about $\ln(10^{6})/\ln(1/0.98) \approx 690$ steps. Tuned heavy ball contracts by
$(\sqrt\kappa-1)/(\sqrt\kappa+1) = 0.818$ and needs about $70$ — roughly a factor $\sqrt\kappa$ fewer.
(The count you measure in Exercise 9 comes out somewhat higher: with the tuned $\beta$ the extreme modes
are critically damped, which adds a factor $k$ to $0.818^k$; the speed-up is still of order $\sqrt\kappa$, only a constant factor smaller.)

</details>

## Exercise 11 — Diagnose a KKT argument  `★★☆`

A data scientist solves $\min_w \tfrac12\lVert w\rVert^2$ subject to $y_i(w^\top x_i + b) \ge 1$ for all $i$ (the
hard-margin SVM) and reports: *"All 10 000 training points have a positive Lagrange multiplier, so
all of them are support vectors, and the model is therefore very robust."* Using the KKT conditions,
explain why this report is almost certainly wrong, what the multipliers of most points must be, and
what a positive multiplier tells you about a point's position relative to the margin.

<details><summary><b>Show answer</b></summary>

Write the constraints as $g_i(w, b) = 1 - y_i(w^\top x_i + b) \le 0$ with multipliers $\alpha_i \ge 0$.
Complementary slackness, $\alpha_i\, g_i = 0$, says that a point with $\alpha_i > 0$ must have
$g_i = 0$, i.e. it lies **exactly on the margin** $y_i(w^\top x_i + b) = 1$. A point strictly outside
the margin has $g_i < 0$ and hence $\alpha_i = 0$. For 10 000 points in general position it is
impossible for all of them to lie exactly on two parallel hyperplanes, so the vast majority must
have $\alpha_i = 0$ — they are not support vectors and could be deleted without changing the
solution $w = \sum_i \alpha_i y_i x_i$. If the solver really returned $\alpha_i > 0$ for every
point, either the multipliers are tiny numerical noise (check them against a tolerance) or a
soft-margin formulation was used, where $\alpha_i = C$ for every *misclassified* or
inside-margin point — which would indicate a badly fitting model, the opposite of "robust".
A positive multiplier means the constraint is active and "pushing": removing that point would
move the decision boundary.

</details>

## Exercise 12 — Mini-project: GD vs momentum vs Adam on logistic regression  `★★★`

Generate a two-class dataset with 400 points in 10 dimensions where the feature scales differ a lot
(so that the problem is badly conditioned), for example

```python
r = np.random.default_rng(1)
scales = np.logspace(0, 1.5, 10)                 # feature scales from 1 to ~32
X = r.normal(size=(400, 10)) * scales
w_true = r.normal(size=10) / scales             # so every feature matters about equally
y = (X @ w_true + 0.5 * r.normal(size=400) > 0).astype(float)
```

Train an $\ell_2$-regularised logistic regression (intercept included, $\lambda = 10^{-3}$) with
**plain GD, heavy-ball momentum ($\beta = 0.9$) and Adam**, each implemented by you, for 500
full-batch steps, and

1. plot the training loss curves of the three methods on a log scale, for at least three learning
   rates per method (e.g. a factor 3 apart);
2. report the final loss and training accuracy of the best run of each method;
3. repeat the experiment with standardised features and compare;
4. write 3–5 sentences on which method is most sensitive to the learning rate, how standardisation
   changes the picture, and what you would recommend to a colleague fitting such a model.

**Rubric — you have mastered this when you can:**

* implement the three optimisers correctly from their update equations (no library optimisers);
* choose learning-rate grids that include at least one clearly too-large (oscillating or diverging) and one too-slow run per method, and recognise them in the plot;
* identify the raw problem's bad conditioning as the reason plain GD is slow and relate it to the feature scales;
* show that standardisation narrows the gap between the methods and permits a larger stable step;
* report accuracy honestly (training accuracy on the same data is optimistic) and say so;
* give a recommendation that mentions both feature scaling and a learning-rate test.

In [ ]:
# Your mini-project workspace

## Score

In [ ]:
summary()